# Лабораторна робота № 1. Тема 12
## Створення вебінтерфейсів із віджетами Streamlit

**Виконав:** Кривенко Назар Володимирович

**Мета.** Створити застосунок зі студентською анкетою, подіями віджетів,
редактором CSV, дашбордом продажів, опитуванням, моделлю прогнозування
та генератором документів.

1. Виконайте три клітинки по черзі або виберіть **Середовище виконання → Виконати всі**.
2. У виході третьої клітинки відкриється інтерфейс. Завдання доступні в бічному меню.
3. Спробуйте анкету з вигаданим ім’ям, CSV-редактор із файлом
   `streamlit_lab/sales.csv` та прогноз для **120 хв і 3D**. Очікувана ціна — **160,00 грн**.

Якщо Colab не відкриє панель застосунку, у вкладці **Файли** завантажте
`streamlit_lab_project.zip`, розпакуйте його та виконайте з теки проєкту
`python -m pip install -r requirements.txt` і `streamlit run app.py`.
Щоб зберегти результати до завершення сеансу Colab, завантажте потрібні файли
з теки `streamlit_lab/data`.

In [ ]:
# Підготувати файли проєкту та встановити бібліотеки.
import subprocess
import sys
from pathlib import Path

project = Path.cwd() / "streamlit_lab"
project.mkdir(exist_ok=True)
(project / ".streamlit").mkdir(exist_ok=True)
(project / 'sales.csv').write_text('date,category,revenue\n2026-11-01,Квитки 2D,130\n2026-11-01,Квитки 3D,160\n2026-11-01,Бар,95\n2026-11-02,Квитки 2D,260\n2026-11-02,Квитки 3D,175\n2026-11-02,Бар,120\n2026-11-03,Квитки 2D,150\n2026-11-03,Квитки 3D,320\n2026-11-03,Бар,90\n2026-11-04,Квитки 2D,280\n2026-11-04,Квитки 3D,190\n2026-11-04,Бар,145\n2026-11-05,Квитки 2D,135\n2026-11-05,Квитки 3D,170\n2026-11-05,Бар,110\n2026-11-06,Квитки 2D,300\n2026-11-06,Квитки 3D,360\n2026-11-06,Бар,155\n', encoding='utf-8')
(project / 'requirements.txt').write_text('streamlit==1.67.0\naltair>=5.5,<7\nnumpy>=1.26,<3\npandas>=2.2,<4\nscikit-learn>=1.5,<2\npython-docx>=1.1,<2\n', encoding='utf-8')
(project / '.streamlit/config.toml').write_text('[theme]\nbase = "light"\nprimaryColor = "#215C83"\n\n[client]\ntoolbarMode = "minimal"\n\n[browser]\ngatherUsageStats = false\n', encoding='utf-8')
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", str(project / "requirements.txt")], check=True)


In [ ]:
%%writefile streamlit_lab/app.py
"""Інтерактивні приклади для лабораторної роботи зі Streamlit."""

import base64
import csv
import hashlib
import json
import sqlite3
import time
from contextlib import closing
from datetime import UTC, datetime
from io import BytesIO, StringIO
from pathlib import Path
from threading import Lock

import altair as alt
import numpy as np
import pandas as pd
import streamlit as st
from docx import Document
from sklearn.linear_model import LinearRegression

ROOT = Path(__file__).resolve().parent
DATA = ROOT / "data"
STUDENT_FIELDS = ("name", "age", "major", "gpa", "dorm", "timestamp")
FEATURES = ["minutes", "is_3d"]


@st.cache_resource
def storage_lock():
    """Повернути спільний для сеансів одного процесу замок запису."""
    return Lock()


def connect_students():
    """Відкрити SQLite та створити таблицю, якщо її ще немає."""
    db = sqlite3.connect(DATA / "students.sqlite3")
    db.execute("""
        CREATE TABLE IF NOT EXISTS students (
            id INTEGER PRIMARY KEY, name TEXT NOT NULL, age INTEGER NOT NULL,
            major TEXT NOT NULL, gpa REAL NOT NULL, dorm INTEGER NOT NULL,
            timestamp TEXT NOT NULL
        )
    """)
    return db


def save_student(record):
    """Додати студента в SQLite та дописати рядок у CSV."""
    path = DATA / "students.csv"
    with storage_lock(), closing(connect_students()) as db, db:
        db.execute("""
            INSERT INTO students (name, age, major, gpa, dorm, timestamp)
            VALUES (:name, :age, :major, :gpa, :dorm, :timestamp)
        """, record)
        write_header = not path.exists() or path.stat().st_size == 0
        with path.open("a+", encoding="utf-8", newline="") as file:
            file.seek(0, 2)
            original_size = file.tell()
            writer = csv.DictWriter(file, fieldnames=STUDENT_FIELDS)
            try:
                if write_header:
                    writer.writeheader()
                writer.writerow(record)
                file.flush()
                db.commit()
            except (OSError, sqlite3.Error):
                file.truncate(original_size)
                raise


def students_page():
    """Показати форму, медіа та вкладку з усіма записами SQLite."""
    form_tab, records_tab = st.tabs(["Анкета", "Записи SQLite"])
    with form_tab:
        name = st.text_input("Ім’я студента")
        age = st.number_input("Вік", min_value=14, max_value=120, value=19)
        major = st.selectbox("Спеціальність", [
            "Кібербезпека", "Право", "Правоохоронна діяльність", "Інша",
        ])
        gpa = st.slider("Середній бал", 0.0, 5.0, 4.0, 0.1)
        dorm = st.checkbox("Проживаю в гуртожитку")
        photo = st.file_uploader("Фото", type=["jpg", "jpeg", "png"])
        media = st.file_uploader("Аудіо або відео", type=["mp3", "wav", "mp4"])
        if media and media.size:
            if media.name.lower().endswith(".mp4"):
                st.video(media)
            else:
                st.audio(media, format="audio/wav" if media.name.lower()
                         .endswith(".wav") else "audio/mpeg")

        if st.button("Надіслати", type="primary"):
            if not name.strip():
                st.warning("Введіть ім’я студента.")
            else:
                progress = st.progress(0, text="Обробка анкети")
                for percent in range(25, 101, 25):
                    time.sleep(0.08)
                    progress.progress(percent)
                record = dict(name=name.strip(), age=int(age), major=major,
                              gpa=gpa, dorm=int(dorm),
                              timestamp=datetime.now(UTC).isoformat(
                                  timespec="seconds"))
                try:
                    save_student(record)
                except (OSError, sqlite3.Error) as error:
                    st.error(f"Помилка запису анкети. {error}")
                else:
                    st.session_state.student_card = record
                    st.session_state.student_photo = (
                        photo.getvalue() if photo else None)
                    st.success("Анкету збережено у CSV та SQLite.")
        if "student_card" in st.session_state:
            card = st.session_state.student_card
            with st.container(border=True):
                st.text(card["name"])
                st.write(f'{card["age"]} р. · {card["major"]}')
                st.write(f'Середній бал — {card["gpa"]:.1f} із 5')
                st.write("Гуртожиток — " + ("так" if card["dorm"] else "ні"))
                st.text(card["timestamp"])
                if st.session_state.student_photo:
                    try:
                        st.image(st.session_state.student_photo, width=220)
                    except (OSError, ValueError):
                        st.warning("Не вдалося прочитати зображення.")
    with records_tab:
        try:
            with closing(connect_students()) as db:
                records = pd.read_sql_query(
                    "SELECT * FROM students ORDER BY id DESC", db)
        except (OSError, sqlite3.Error, pd.errors.DatabaseError) as error:
            st.error(f"Не вдалося прочитати записи. {error}")
        else:
            if records.empty:
                st.info("Анкет поки немає.")
            else:
                st.dataframe(records, hide_index=True)


def events_page():
    """Побудувати графік за кнопкою і продемонструвати стан сеансу."""
    function = st.selectbox("Функція", ["sin", "cos", "exp"])
    amplitude = st.slider("Амплітуда", 0.0, 5.0, 1.0, 0.1)
    frequency = st.slider("Частота або коефіцієнт експоненти", 0.1, 3.0, 1.0, 0.1)
    if st.button("Побудувати графік", type="primary"):
        x = np.linspace(-2 * np.pi, 2 * np.pi, 400)
        operation = {"sin": np.sin, "cos": np.cos, "exp": np.exp}[function]
        st.session_state.graph = pd.DataFrame(
            {"x": x, "y": amplitude * operation(frequency * x)}
        ).set_index("x")
        st.session_state.graph_label = f"{amplitude:g} · {function}({frequency:g}x)"
    if "graph" in st.session_state:
        st.write(st.session_state.graph_label)
        st.line_chart(st.session_state.graph, y_label="y")
    st.divider()
    if st.button("Додати натискання"):
        st.session_state.click_count = st.session_state.get("click_count", 0) + 1
    st.metric("Кількість натискань", st.session_state.get("click_count", 0))

    # URL відновлює останнє число після повного оновлення вкладки.
    if "last_value" not in st.session_state:
        try:
            value = int(st.query_params.get("number", "42"))
        except ValueError:
            value = 42
        st.session_state.last_value = max(0, min(100, value))
    if "event_number" not in st.session_state:
        st.session_state.event_number = st.session_state.last_value
    value = st.number_input("Число для відновлення після оновлення",
                            0, 100, key="event_number")
    st.session_state.last_value = value
    if st.query_params.get("number") != str(value):
        st.query_params["number"] = str(value)


@st.cache_data(max_entries=8)
def read_csv(data):
    """Прочитати завантажений CSV у UTF-8 з комою або крапкою з комою."""
    content = data.decode("utf-8-sig")
    try:
        separator = csv.Sniffer().sniff(content[:8192], delimiters=",;").delimiter
    except csv.Error:
        separator = ","
    table = pd.read_csv(StringIO(content), sep=separator)
    if table.empty:
        raise ValueError("CSV не містить рядків даних.")
    return table


def editor_page():
    """Оновлювати копію відредагованої таблиці у стані сеансу."""
    upload = st.file_uploader("CSV для редагування", type="csv")
    if upload is None:
        st.session_state.pop("edited_table", None)
        return
    data = upload.getvalue()
    try:
        source = read_csv(data)
    except (ValueError, csv.Error) as error:
        st.session_state.pop("edited_table", None)
        st.error(f"Не вдалося прочитати CSV. {error}")
        return
    digest = hashlib.sha256(data).hexdigest()
    # Редактор застосовує зміни до незмінного початкового набору рядків.
    edited = st.data_editor(source, key=f"csv_{digest}", num_rows="dynamic")
    st.session_state.edited_table = edited
    st.caption(f"Рядків у поточній версії — {len(edited)}")
    st.download_button("Завантажити змінений CSV",
                       edited.to_csv(index=False).encode("utf-8-sig"),
                       "edited.csv", "text/csv")


@st.cache_data
def load_sales():
    """Прочитати демонстраційні продажі, де рядок відповідає чеку."""
    return pd.read_csv(ROOT / "sales.csv", parse_dates=["date"])


def sales_page():
    """Застосувати фільтри й показати дохід, середній чек і графіки."""
    sales = load_sales()
    first, last = sales.date.min().date(), sales.date.max().date()
    start = st.sidebar.date_input("Від", first, min_value=first, max_value=last)
    end = st.sidebar.date_input("До", last, min_value=first, max_value=last)
    categories = sorted(sales.category.unique())
    selected = st.sidebar.multiselect("Категорії", categories, default=categories)
    if start > end:
        st.warning("Початкова дата має бути не пізнішою за кінцеву.")
        return
    filtered = sales[sales.date.between(pd.Timestamp(start), pd.Timestamp(end))
                     & sales.category.isin(selected)]
    with st.container():
        left, right = st.columns(2)
        left.metric("Загальний дохід", f"{filtered.revenue.sum():.2f} грн")
        average = filtered.revenue.mean()
        right.metric("Середній чек", f"{average:.2f} грн" if len(filtered) else "—")
        if filtered.empty:
            st.info("За вибраними фільтрами чеків немає.")
            return
        st.subheader("Дохід за днями")
        days = pd.date_range(start, end)
        daily = (filtered.groupby("date").revenue.sum()
                 .reindex(days, fill_value=0).rename_axis("date").reset_index())
        st.altair_chart(alt.Chart(daily).mark_line(point=True).encode(
            x=alt.X("date:T", title="Дата"),
            y=alt.Y("revenue:Q", title="Дохід, грн"),
            tooltip=["date:T", "revenue:Q"],
        ))
        st.subheader("Дохід за категоріями")
        totals = filtered.groupby("category", as_index=False).revenue.sum()
        st.altair_chart(alt.Chart(totals).mark_bar().encode(
            x=alt.X("category:N", title="Категорія"),
            y=alt.Y("revenue:Q", title="Дохід, грн"),
            tooltip=["category:N", "revenue:Q"],
        ))
        st.dataframe(filtered, hide_index=True)


def survey_records():
    """Прочитати список відгуків або повернути порожній список."""
    path = DATA / "survey.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else []


def save_survey(record):
    """Додати відгук і атомарно замінити JSON-файл."""
    with storage_lock():
        records = survey_records()
        records.append(record)
        temporary = DATA / "survey.tmp"
        temporary.write_text(json.dumps(records, ensure_ascii=False, indent=2),
                             encoding="utf-8")
        temporary.replace(DATA / "survey.json")


def survey_page():
    """Зібрати оцінку курсу, текст, варіанти вибору й вкладення."""
    with st.form("survey", clear_on_submit=True):
        name = st.text_input("Ім’я")
        comment = st.text_area("Відгук про курс")
        rating = st.slider("Оцінка", 1, 5, 5)
        topics = st.multiselect("Що було корисним?",
                                ["Теорія", "Практика", "Приклади коду"])
        attachment = st.file_uploader("Додати відгук", type=["txt", "pdf"])
        submitted = st.form_submit_button("Зберегти відгук")
    try:
        if submitted:
            if not name.strip() or not comment.strip():
                st.warning("Заповніть ім’я та відгук.")
            elif attachment is not None and not attachment.size:
                st.warning("Файл відгуку порожній.")
            elif attachment is not None and attachment.size > 2 * 1024 * 1024:
                st.warning("Вкладення має бути до 2 МБ.")
            else:
                # Base64 дає змогу зберегти вкладення в тому самому JSON.
                file = None
                if attachment is not None:
                    file = {"name": attachment.name, "data_base64":
                            base64.b64encode(attachment.getvalue()).decode("ascii")}
                save_survey(dict(name=name.strip(), comment=comment.strip(),
                                 rating=rating, topics=topics, attachment=file))
                st.success("Відгук збережено.")
        st.metric("Кількість відгуків", len(survey_records()))
    except (OSError, ValueError) as error:
        st.error(f"Помилка роботи з відгуками. {error}")


@st.cache_resource
def train_model():
    """Навчити регресію на вигаданих цінах квитків у форматах 2D і 3D."""
    samples = pd.DataFrame(
        [(minutes, is_3d) for minutes in range(60, 181, 20) for is_3d in (0, 1)],
        columns=FEATURES,
    )
    prices = 70 + 0.5 * samples.minutes + 30 * samples.is_3d
    return LinearRegression().fit(samples, prices)


def prediction_page():
    """Отримати прогноз ціни для заданої тривалості й формату."""
    st.caption("Модель навчено на вигаданих цінах квитків.")
    minutes = st.slider("Тривалість фільму, хв", 60, 180, 120)
    movie_format = st.selectbox("Формат", ["2D", "3D"])
    if st.button("Передбачити ціну", type="primary"):
        features = pd.DataFrame([[minutes, int(movie_format == "3D")]],
                                columns=FEATURES)
        price = train_model().predict(features)[0]
        st.success(f"Прогнозована ціна — {price:.2f} грн")


def make_docx(name, position, date, reason):
    """Сформувати заяву Word і повернути її байти."""
    doc = Document()
    doc.add_heading("Заява", 0)
    doc.add_paragraph(f"ПІБ — {name}")
    doc.add_paragraph(f"Посада — {position}")
    doc.add_paragraph(f"Прошу розглянути мою заяву з такої причини: {reason}")
    doc.add_paragraph(f"Дата — {date:%d.%m.%Y}")
    doc.add_paragraph("Підпис ____________________")
    buffer = BytesIO()
    doc.save(buffer)
    return buffer.getvalue()


def documents_page():
    """Прийняти реквізити та запропонувати завантажити готовий DOCX."""
    with st.form("document"):
        name = st.text_input("ПІБ")
        position = st.text_input("Посада")
        date = st.date_input("Дата", format="DD.MM.YYYY")
        reason = st.text_area("Причина")
        submitted = st.form_submit_button("Згенерувати")
    if submitted:
        st.session_state.pop("document_bytes", None)
        if not all(value.strip() for value in (name, position, reason)):
            st.warning("Заповніть ПІБ, посаду та причину.")
        else:
            st.session_state.document_bytes = make_docx(
                name.strip(), position.strip(), date, reason.strip())
    if "document_bytes" in st.session_state:
        st.download_button(
            "Завантажити заяву", st.session_state.document_bytes, "zaiava.docx",
            "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        )


def intro_page():
    """Показати заголовок, підзаголовок, абзац і формулу Ейлера."""
    st.title("Лабораторна робота з Streamlit")
    st.subheader("Створення вебінтерфейсів із віджетами")
    st.write("Виберіть розділ у бічному меню, щоб перевірити роботу віджетів.")
    st.latex(r"e^{i\pi}+1=0")


def main():
    """Налаштувати застосунок і відкрити вибраний розділ."""
    st.set_page_config(page_title="Лабораторна робота зі Streamlit",
                       layout="wide")
    DATA.mkdir(exist_ok=True)
    pages = {
        "1 · Вступ": intro_page,
        "2–3 · Анкета студента": students_page,
        "4 · Графік та події": events_page,
        "4.3 · Редактор CSV": editor_page,
        "5.1 · Продажі кінотеатру": sales_page,
        "5.2 · Опитування": survey_page,
        "5.3 · Прогноз ціни": prediction_page,
        "5.4 · Генератор заяви": documents_page,
    }
    if "page" not in st.session_state:
        requested = st.query_params.get("page")
        if requested in pages:
            st.session_state.page = requested
    page = st.sidebar.radio("Розділ лабораторної", list(pages), key="page")
    if st.query_params.get("page") != page:
        st.query_params["page"] = page
    if page != "1 · Вступ":
        st.title(page.split(" · ", 1)[1])
    pages[page]()


if __name__ == "__main__":
    main()


In [ ]:
# Створити ZIP для локального запуску і запустити Streamlit у фоні.
import time
import zipfile
from urllib.error import URLError
from urllib.request import urlopen

archive = project.parent / "streamlit_lab_project.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for filename in ("app.py", "sales.csv", "requirements.txt",
                     ".streamlit/config.toml"):
        bundle.write(project / filename, filename)

if "server" in globals() and server.poll() is None:
    server.terminate()
    try:
        server.wait(timeout=8)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()

port = 8501
log_path = project / "streamlit.log"
with log_path.open("w", encoding="utf-8") as log:
    server = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", "app.py",
         "--server.address=127.0.0.1", f"--server.port={port}",
         "--server.headless=true"],
        cwd=project, stdout=log, stderr=subprocess.STDOUT,
    )

deadline = time.monotonic() + 35
while time.monotonic() < deadline and server.poll() is None:
    try:
        with urlopen(f"http://127.0.0.1:{port}/_stcore/health", timeout=1) as response:
            if response.status == 200:
                break
    except (OSError, URLError):
        time.sleep(0.25)
else:
    raise RuntimeError(log_path.read_text(encoding="utf-8", errors="replace"))

try:
    from google.colab import output
except ImportError:
    print(f"Відкрийте http://127.0.0.1:{port}")
else:
    output.serve_kernel_port_as_iframe(port, height=840)


### Як працює повторний запуск

Streamlit після взаємодії з більшістю віджетів знову виконує скрипт згори
вниз. Кнопка повертає `True` на запуску, спричиненому натисканням.
`st.session_state` зберігає лічильник, графік і відредаговані дані між
такими запусками. У формах опитування й генератора введені значення
надсилаються разом після підтвердження. Під час повного оновлення вкладки
створюється новий сеанс. Для відновлення останнього числа застосунок
зчитує параметр `number` з адреси. У Flask HTTP-запит обробляє
функція відповідного маршруту, яка формує відповідь.

[Стан сеансу Streamlit](https://docs.streamlit.io/develop/concepts/architecture/session-state) ·
[Життєвий цикл запиту у Flask](https://flask.palletsprojects.com/en/stable/lifecycle/)